In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Soret\insurance-risk-analytics\data\cleaned_data.csv")
df.head()

C:\Users\Soret\AppData\Local\Temp\ipykernel_3904\2190413666.py:3: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\Soret\insurance-risk-analytics\data\cleaned_data.csv")


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


In [3]:
df["has_claim"] = (df["TotalClaims"] > 0).astype(int)

df["claim_frequency"] = df["has_claim"]

df["claim_severity"] = df["TotalClaims"]
df.loc[df["TotalClaims"] == 0, "claim_severity"] = None

df["margin"] = df["TotalPremium"] - df["TotalClaims"]

In [4]:
from scipy.stats import chi2_contingency, ttest_ind

def chi_square_test(data, feature, target):
    table = pd.crosstab(data[feature], data[target])
    chi2, p, dof, exp = chi2_contingency(table)
    return p

def t_test(data, group_col, value_col, g1, g2):
    a = data[data[group_col] == g1][value_col].dropna()
    b = data[data[group_col] == g2][value_col].dropna()
    stat, p = ttest_ind(a, b, equal_var=False)
    return p

In [5]:
group_A_province = df["Province"].value_counts().idxmax()

provinces = [p for p in df["Province"].unique() if p != group_A_province]

group_B_province = min(
    provinces,
    key=lambda x: abs(len(df[df["Province"] == x]) - len(df[df["Province"] == group_A_province]))
)

In [6]:
group_A_zip = df["PostalCode"].value_counts().idxmax()

zips = [z for z in df["PostalCode"].unique() if z != group_A_zip]

group_B_zip = min(
    zips,
    key=lambda x: abs(len(df[df["PostalCode"] == x]) - len(df[df["PostalCode"] == group_A_zip]))
)

In [7]:
group_A_gender = df["Gender"].value_counts().idxmax()
group_B_gender = [g for g in df["Gender"].unique() if g != group_A_gender][0]

In [8]:
p_province = chi_square_test(df, "Province", "has_claim")

In [9]:
p_zip_freq = chi_square_test(df, "PostalCode", "has_claim")

In [10]:
p_zip_margin = t_test(df, "PostalCode", "margin", group_A_zip, group_B_zip)

In [11]:
def decision(p):
    return "Reject H0" if p < 0.05 else "Fail to reject H0"

In [13]:
p_gender = chi_square_test(df, "Gender", "has_claim")
p_gender

np.float64(0.026570248768437145)

In [14]:
results = pd.DataFrame([
    ["Province vs Risk", "Claim Frequency", p_province, decision(p_province)],
    ["Zip vs Risk", "Claim Frequency", p_zip_freq, decision(p_zip_freq)],
    ["Zip vs Margin", "Margin", p_zip_margin, decision(p_zip_margin)],
    ["Gender vs Risk", "Claim Frequency", p_gender, decision(p_gender)]
], columns=["Hypothesis", "KPI", "p-value", "Decision"])

results

,Hypothesis,KPI,p-value,Decision
0,Province vs Risk,Claim Frequency,5.925511e-19,Reject H0
1,Zip vs Risk,Claim Frequency,3.152172e-30,Reject H0
2,Zip vs Margin,Margin,2.444624e-01,Fail to reject H0
3,Gender vs Risk,Claim Frequency,2.657025e-02,Reject H0


In [15]:
df["has_claim"] = (df["TotalClaims"] > 0).astype(int)

df["claim_frequency"] = df["has_claim"]

df["claim_severity"] = df["TotalClaims"]
df.loc[df["TotalClaims"] == 0, "claim_severity"] = None

df["margin"] = df["TotalPremium"] - df["TotalClaims"]

In [16]:
from scipy.stats import chi2_contingency, ttest_ind
import pandas as pd

In [17]:
def chi_square_pvalue(df, feature, target):
    table = pd.crosstab(df[feature], df[target])
    chi2, p, dof, exp = chi2_contingency(table)
    return p


def t_test_pvalue(df, group_col, value_col, group_a, group_b):
    a = df[df[group_col] == group_a][value_col].dropna()
    b = df[df[group_col] == group_b][value_col].dropna()
    stat, p = ttest_ind(a, b, equal_var=False)
    return p

In [18]:
p_province = chi_square_pvalue(df, "Province", "has_claim")

In [19]:
p_zip_freq = chi_square_pvalue(df, "PostalCode", "has_claim")

In [20]:
group_A_zip = df["PostalCode"].value_counts().idxmax()
group_B_zip = df["PostalCode"].value_counts().index[1]

In [21]:
p_zip_margin = t_test_pvalue(df, "PostalCode", "margin", group_A_zip, group_B_zip)

In [22]:
p_gender = chi_square_pvalue(df, "Gender", "has_claim")

In [23]:
def decision(p):
    return "Reject H0" if p < 0.05 else "Fail to reject H0"

In [24]:
results = pd.DataFrame([
    ["Province vs Risk", "Claim Frequency", p_province, decision(p_province)],
    ["Zip vs Risk", "Claim Frequency", p_zip_freq, decision(p_zip_freq)],
    ["Zip vs Margin", "Margin", p_zip_margin, decision(p_zip_margin)],
    ["Gender vs Risk", "Claim Frequency", p_gender, decision(p_gender)]
], columns=["Hypothesis", "KPI", "p-value", "Decision"])

results

,Hypothesis,KPI,p-value,Decision
0,Province vs Risk,Claim Frequency,5.925511e-19,Reject H0
1,Zip vs Risk,Claim Frequency,3.152172e-30,Reject H0
2,Zip vs Margin,Margin,2.444624e-01,Fail to reject H0
3,Gender vs Risk,Claim Frequency,2.657025e-02,Reject H0


In [26]:
def decision(p):
    return "Reject H0" if p < 0.05 else "Fail to reject H0"

In [32]:
def business_interpretation(hypothesis, p, group_a=None, group_b=None):
    
    if p >= 0.05:
        return None  # no interpretation for non-rejected hypotheses

    if hypothesis == "Province":
        return f"We reject H₀ for provinces (p < {p:.3f}). Geographic variation in claim frequency suggests meaningful regional risk differences, supporting province-level premium adjustments."

    if hypothesis == "Zip_Frequency":
        return f"We reject H₀ for zip codes (p < {p:.3f}). Claim frequency differs across postal codes, indicating strong micro-geographic risk segmentation opportunities."

    if hypothesis == "Zip_Margin":
        return f"We reject H₀ for zip codes (p < {p:.3f}). Profitability varies significantly across zip codes, indicating pricing misalignment and need for localized risk loadings."

    if hypothesis == "Gender":
        return f"We reject H₀ for gender (p < {p:.3f}). Claim frequency differs between genders, indicating potential predictive value for underwriting (subject to regulatory constraints)."

    return f"We reject H₀ for {hypothesis} (p < {p:.3f}). Significant differences observed, suggesting segmentation is warranted."

In [28]:
from scipy.stats import chi2_contingency, ttest_ind
import pandas as pd

In [29]:
# Province (Chi-square)
table = pd.crosstab(df["Province"], df["has_claim"])
_, p_province, _, _ = chi2_contingency(table)

# Zip frequency (Chi-square)
table = pd.crosstab(df["PostalCode"], df["has_claim"])
_, p_zip_freq, _, _ = chi2_contingency(table)

# Zip margin (t-test)
zips = df["PostalCode"].unique()
g1, g2 = zips[0], zips[1]

_, p_zip_margin = ttest_ind(
    df[df["PostalCode"] == g1]["margin"],
    df[df["PostalCode"] == g2]["margin"],
    equal_var=False
)

# Gender (Chi-square)
table = pd.crosstab(df["Gender"], df["has_claim"])
_, p_gender, _, _ = chi2_contingency(table)

In [30]:
results = [
    ("Province", p_province),
    ("Zip_Frequency", p_zip_freq),
    ("Zip_Margin", p_zip_margin),
    ("Gender", p_gender)
]

final_output = []

for h, p in results:
    final_output.append({
        "Hypothesis": h,
        "p-value": p,
        "Decision": decision(p),
        "Business Insight": business_interpretation(h, p)
    })

pd.DataFrame(final_output)

,Hypothesis,p-value,Decision,Business Insight
0,Province,5.925511e-19,Reject H0,We reject H₀ for provinces (p < 0.000). Geogra...
1,Zip_Frequency,3.152172e-30,Reject H0,We reject H₀ for zip codes (p < 0.000). Claim ...
2,Zip_Margin,6.630316e-01,Fail to reject H0,NaN
3,Gender,2.657025e-02,Reject H0,We reject H₀ for gender (p < 0.027). Claim fre...


In [35]:
results = [
    ["Province vs Risk", "Chi-square", p_province, decision(p_province)],
    ["Zip vs Risk", "Chi-square", p_zip_freq, decision(p_zip_freq)],
    ["Zip vs Margin", "t-test", p_zip_margin, decision(p_zip_margin)],
    ["Gender vs Risk", "Chi-square", p_gender, decision(p_gender)]
]

In [36]:
import pandas as pd

results = pd.DataFrame(results, columns=[
    "Hypothesis", "Test", "p-value", "Decision"
])

In [37]:
results["Business Recommendation"] = results.apply(
    lambda row: business_note(row["Hypothesis"], row["p-value"]),
    axis=1
)

results

NameError: name 'business_note' is not defined

In [38]:
import pandas as pd
from scipy.stats import chi2_contingency, ttest_ind

In [39]:
df = pd.read_csv(r"C:\Users\Soret\insurance-risk-analytics\data\cleaned_data.csv")
df.head()

C:\Users\Soret\AppData\Local\Temp\ipykernel_3904\874842892.py:1: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\Soret\insurance-risk-analytics\data\cleaned_data.csv")


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


In [40]:
df["has_claim"] = (df["TotalClaims"] > 0).astype(int)
df["margin"] = df["TotalPremium"] - df["TotalClaims"]

In [41]:
def chi_square_pvalue(data, feature, target):
    table = pd.crosstab(data[feature], data[target])
    _, p, _, _ = chi2_contingency(table)
    return p


def ttest_pvalue(data, group_col, value_col, g1, g2):
    a = data[data[group_col] == g1][value_col].dropna()
    b = data[data[group_col] == g2][value_col].dropna()
    _, p = ttest_ind(a, b, equal_var=False)
    return p


def decision(p):
    return "Reject H0" if p < 0.05 else "Fail to reject H0"

In [42]:
province_A = df["Province"].value_counts().idxmax()
province_B = df["Province"].value_counts().index[1]

zip_A = df["PostalCode"].value_counts().idxmax()
zip_B = df["PostalCode"].value_counts().index[1]

In [43]:
p_province = chi_square_pvalue(df, "Province", "has_claim")

p_zip_freq = chi_square_pvalue(df, "PostalCode", "has_claim")

p_zip_margin = ttest_pvalue(df, "PostalCode", "margin", zip_A, zip_B)

p_gender = chi_square_pvalue(df, "Gender", "has_claim")

In [44]:
results = pd.DataFrame([
    ["Province vs Risk", "Chi-square", p_province, decision(p_province)],
    ["Zip vs Risk", "Chi-square", p_zip_freq, decision(p_zip_freq)],
    ["Zip vs Margin", "t-test", p_zip_margin, decision(p_zip_margin)],
    ["Gender vs Risk", "Chi-square", p_gender, decision(p_gender)]
], columns=["Hypothesis", "Test", "p-value", "Decision"])

In [45]:
def business_note(row):
    if row["Decision"] == "Fail to reject H0":
        return None

    if "Province" in row["Hypothesis"]:
        return "Geographic risk variation detected. Introduce province-based pricing."

    if "Zip vs Risk" in row["Hypothesis"]:
        return "Strong micro-geographic risk differences. Consider zip-level segmentation."

    if "Zip vs Margin" in row["Hypothesis"]:
        return "Profitability varies across zip codes. Recalibrate pricing locally."

    if "Gender" in row["Hypothesis"]:
        return "Gender shows statistically significant risk difference. Check regulatory constraints."

    return "Significant difference detected."

In [46]:
results["Business Recommendation"] = results.apply(business_note, axis=1)

In [47]:
results = [
    ("Province", p_province),
    ("Zip_Frequency", p_zip_freq),
    ("Zip_Margin", p_zip_margin),
    ("Gender", p_gender)
]

final_output = []

for h, p in results:
    final_output.append({
        "Hypothesis": h,
        "p-value": p,
        "Decision": decision(p),
        "Business Insight": business_interpretation(h, p)
    })

pd.DataFrame(final_output)

,Hypothesis,p-value,Decision,Business Insight
0,Province,5.925511e-19,Reject H0,We reject H₀ for provinces (p < 0.000). Geogra...
1,Zip_Frequency,3.152172e-30,Reject H0,We reject H₀ for zip codes (p < 0.000). Claim ...
2,Zip_Margin,2.444624e-01,Fail to reject H0,NaN
3,Gender,2.657025e-02,Reject H0,We reject H₀ for gender (p < 0.027). Claim fre...


In [48]:
results = pd.DataFrame([
    ["Province vs Risk", "Claim Frequency", p_province, decision(p_province)],
    ["Zip vs Risk", "Claim Frequency", p_zip_freq, decision(p_zip_freq)],
    ["Zip vs Margin", "Margin", p_zip_margin, decision(p_zip_margin)],
    ["Gender vs Risk", "Claim Frequency", p_gender, decision(p_gender)]
], columns=["Hypothesis", "KPI", "p-value", "Decision"])

results

,Hypothesis,KPI,p-value,Decision
0,Province vs Risk,Claim Frequency,5.925511e-19,Reject H0
1,Zip vs Risk,Claim Frequency,3.152172e-30,Reject H0
2,Zip vs Margin,Margin,2.444624e-01,Fail to reject H0
3,Gender vs Risk,Claim Frequency,2.657025e-02,Reject H0


In [49]:
# ============================================
# TASK 3 — A/B HYPOTHESIS TESTING PIPELINE
# ============================================

import pandas as pd
import numpy as np

from hypothesis_tests import (
    chi_square_test,
    t_test,
    anova_test,
    decision_rule
)

# ============================================
# 1. LOAD DATA
# ============================================

df = pd.read_csv("data/cleaned_data.csv")


# ============================================
# 2. CREATE KPI VARIABLES
# ============================================

# Claim Frequency
# 1 = at least one claim
# 0 = no claim

df["has_claim"] = (df["TotalClaims"] > 0).astype(int)


# Claim Severity
# average claim amount ONLY where claim occurred

df["claim_severity"] = np.where(
    df["TotalClaims"] > 0,
    df["TotalClaims"],
    np.nan
)


# Margin
# premium - claims

df["margin"] = df["TotalPremium"] - df["TotalClaims"]


# ============================================
# 3. GROUP SELECTION
# ============================================

# ----------------------------
# Province groups
# ----------------------------

province_counts = df["Province"].value_counts()

group_A_province = province_counts.index[0]
group_B_province = province_counts.index[1]


# ----------------------------
# Zip code groups
#

C:\Users\Soret\AppData\Local\Temp\ipykernel_3904\1310801695.py:19: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/cleaned_data.csv")
